# GraphPPI Demo

## 图神经网络蛋白质互作预测

本 notebook 演示如何使用 GraphPPI 进行 PPI 链接预测：
1. 加载预处理数据
2. 训练一个 GCN+MLP 模型
3. 评估 AUC / AP / Hits@K 指标
4. 可视化基准对比结果

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from graphppi.utils import split_edges_kfold, prepare_fold_data
from graphppi.models.predictor import LinkPredictor
from graphppi.trainer import LinkPredictionTrainer
from graphppi.metrics import compute_all_metrics

print('✅ Imports OK')

## 1. 加载数据

In [ ]:
data = torch.load('data/processed/graph.pt', weights_only=False)
print(f'Nodes: {data.num_nodes}, Directed edges: {data.num_edges}')
print(f'Undirected edges: {data.num_edges // 2}')
print(f'Edge attr shape: {list(data.edge_attr.shape)}')
print(f'First 5 node names: {data.node_names[:5]}')

## 2. 准备数据（单 fold）

In [ ]:
num_undirected = data.edge_index.size(1) // 2
folds = split_edges_kfold(data.edge_index, num_undirected, k=5, seed=42)
fold_data = prepare_fold_data(data, folds[0])

print(f'Train edges: {fold_data["train_edges"].size(1)}')
print(f'Val edges:   {fold_data["val_edges"].size(1)}')
print(f'Test edges:  {fold_data["test_edges"].size(1)}')
print(f'Node features: {list(fold_data["x"].shape)}')

## 3. 训练 GCN-MLP 模型

In [ ]:
x = fold_data['x']
model = LinkPredictor(
    in_dim=x.size(1), encoder_type='gcn', decoder_type='mlp',
    hidden_dim=128, out_dim=64, num_layers=2, dropout=0.5
)
trainer = LinkPredictionTrainer(model, lr=0.005, weight_decay=1e-4)
trainer.train(
    x=x,
    mp_edge_index=fold_data['mp_edge_index'],
    train_edges=fold_data['train_edges'],
    train_labels=fold_data['train_labels'],
    val_edges=fold_data['val_edges'],
    val_labels=fold_data['val_labels'],
    epochs=200, patience=20,
    mp_edge_weight=fold_data['mp_edge_weight'],
    train_edge_attr=fold_data['train_edge_attr'],
    val_edge_attr=fold_data['val_edge_attr'],
)
print(f'Best val AUC: {trainer.best_val_auc:.4f}')

## 4. 评估结果

In [ ]:
test_metrics = trainer.test(
    x=x,
    mp_edge_index=fold_data['mp_edge_index'],
    test_edges=fold_data['test_edges'],
    test_labels=fold_data['test_labels'],
    mp_edge_weight=fold_data['mp_edge_weight'],
    test_edge_attr=fold_data['test_edge_attr'],
)

for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

## 5. 基准对比可视化

In [ ]:
methods = ['GraphSAGE\n+ MLP', 'GCN\n+ MLP', 'GAT\n+ MLP', 'Adamic-Adar',
           'GCN\n+ Dot', 'Common\nNeighbors', 'Jaccard', 'Node2Vec\n+ RF']
auc_values = [0.9399, 0.9257, 0.9150, 0.9035, 0.8978, 0.8969, 0.8884, 0.8801]
ap_values  = [0.9425, 0.9246, 0.9098, 0.8798, 0.9048, 0.8684, 0.8598, 0.8565]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(methods)); width = 0.35
ax.bar(x-width/2, auc_values, width, label='AUC', color='#2196F3', alpha=0.85)
ax.bar(x+width/2, ap_values, width, label='AP', color='#4CAF50', alpha=0.85)
ax.axhline(y=0.9035, color='red', linestyle='--', label='Adamic-Adar baseline')
for bar, val in zip(ax.patches[:8], auc_values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.4f}', ha='center', fontsize=7, fontweight='bold')
ax.set_ylabel('Score'); ax.set_title('GraphPPI Link Prediction (3-fold CV)')
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=8)
ax.legend(loc='lower right'); ax.set_ylim(0.82, 0.98); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 6. 结论

- 🥇 **GraphSAGE + MLP** 是公平对比下最佳模型（AUC 0.940）
- MLP 解码器是最大单次增益（+3.7% vs Dot）
- GNN（SAGE/GCN）全面超越传统拓扑基线
- STRING 8 通道边特征可作为外部知识辅助预测（参考 AUC 0.999）